# Pengujian Model Serving (Prediction Request): Heart Disease Prediction
- **Nama:** M. Rizal Basri
- **Username Dicoding:** rizalbasri
- **Model:** Heart Disease Classification Model (TensorFlow Serving SavedModel)
- **Tujuan:** Menguji dan melakukan prediction request ke model serving yang telah dibuat pada pipeline TFX.

---
## Deskripsi Pengujian
Notebook ini dibuat untuk memenuhi **Saran 3 (Kriteria Bintang 5)** submission Dicoding:
> *"Menambahkan sebuah berkas notebook untuk menguji dan melakukan prediction request ke model serving yang telah dibuat. Berkas notebook ini harus bernama `<username_dicoding>-testing.ipynb`."*

Pengujian mencakup:
1. Mempersiapkan data uji dari observasi klinis pasien.
2. Melakukan serialisasi fitur menjadi format `tf.train.Example`.
3. Menguji model serving signature (`serving_default`) langsung dari SavedModel di direktori `serving_model_dir`.
4. Mengirimkan prediction request menggunakan format REST API TensorFlow Serving (JSON payload dengan base64 encoded TFRecord).
5. Memverifikasi hasil probabilitas dan klasifikasi biner terhadap diagnosis aktual.


## 1. Import Library
Mengimpor pustaka yang diperlukan untuk memproses data, serialisasi tf.train.Example, dan mengirimkan prediction request.


In [1]:
import os
import json
import base64
import numpy as np
import pandas as pd
import tensorflow as tf
import requests

print(f"TensorFlow Version: {tf.__version__}")


D:\Coding\dicoding\M_Rizal_Basri-pipeline\.venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


TensorFlow Version: 2.13.1


## 2. Memilih Sampel Data Uji Pasien
Kita mengambil beberapa sampel observasi pasien dari dataset `data/heart.csv` untuk diuji:
- **Pasien A:** Pasien dengan diagnosis aktual memiliki penyakit jantung (`target = 1`).
- **Pasien B:** Pasien dengan diagnosis aktual tidak memiliki penyakit jantung / sehat (`target = 0`).


In [2]:
df = pd.read_csv(os.path.join("data", "heart.csv"))

# Ambil sampel pasien positif (target=1) dan negatif (target=0)
sample_positive = df[df['target'] == 1].iloc[0].to_dict()
sample_negative = df[df['target'] == 0].iloc[0].to_dict()

samples = [sample_positive, sample_negative]
print("Sampel Data Uji:")
for i, s in enumerate(samples):
    print(f"\n--- Pasien {i+1} (Aktual Target: {int(s['target'])}) ---")
    for k, v in list(s.items())[:7]:
        print(f"  {k}: {v}")


Sampel Data Uji:

--- Pasien 1 (Aktual Target: 1) ---
  age: 63.0
  sex: 1.0
  cp: 3.0
  trestbps: 145.0
  chol: 233.0
  fbs: 1.0
  restecg: 0.0

--- Pasien 2 (Aktual Target: 0) ---
  age: 67.0
  sex: 1.0
  cp: 0.0
  trestbps: 160.0
  chol: 286.0
  fbs: 0.0
  restecg: 0.0


## 3. Serialisasi Data Uji ke Format `tf.train.Example`
Karena model diekspor dengan serving signature yang menerima raw serialized `tf.train.Example`, kita mengonversi input data uji pasien menjadi format TFRecord `tf.train.Example`.


In [3]:
def _int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[int(value)]))

def _float_feature(value):
    return tf.train.Feature(float_list=tf.train.FloatList(value=[float(value)]))

def create_serialized_tf_example(data_dict):
    feature = {
        "age": _int64_feature(data_dict["age"]),
        "sex": _int64_feature(data_dict["sex"]),
        "cp": _int64_feature(data_dict["cp"]),
        "trestbps": _int64_feature(data_dict["trestbps"]),
        "chol": _int64_feature(data_dict["chol"]),
        "fbs": _int64_feature(data_dict["fbs"]),
        "restecg": _int64_feature(data_dict["restecg"]),
        "thalach": _int64_feature(data_dict["thalach"]),
        "exang": _int64_feature(data_dict["exang"]),
        "oldpeak": _float_feature(data_dict["oldpeak"]),
        "slope": _int64_feature(data_dict["slope"]),
        "ca": _int64_feature(data_dict["ca"]),
        "thal": _int64_feature(data_dict["thal"])
    }
    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return example.SerializeToString()

serialized_examples = [create_serialized_tf_example(s) for s in samples]
print(f"Berhasil membuat {len(serialized_examples)} serialized tf.train.Example instances.")


Berhasil membuat 2 serialized tf.train.Example instances.


## 4. Pengujian 1: Memuat SavedModel Serving dan Inferensi Langsung
Kita memuat model dari direktori `serving_model_dir` yang telah diekspor oleh komponen `Pusher` pada pipeline TFX, lalu menguji signature `serving_default`.


In [4]:
# Temukan model versi terbaru di serving_model_dir
SERVING_DIR = "serving_model_dir"
versions = sorted([v for v in os.listdir(SERVING_DIR) if os.path.isdir(os.path.join(SERVING_DIR, v))])
latest_version = versions[-1]
model_path = os.path.join(SERVING_DIR, latest_version)

print(f"Memuat Model Serving dari: {model_path}")
loaded_model = tf.saved_model.load(model_path)
serving_fn = loaded_model.signatures["serving_default"]

print("Serving Signature Inputs:", serving_fn.structured_input_signature)
print("Serving Signature Outputs:", serving_fn.structured_outputs)


Memuat Model Serving dari: serving_model_dir\1789440851


Serving Signature Inputs: ((), {'examples': TensorSpec(shape=(None,), dtype=tf.string, name='examples')})
Serving Signature Outputs: {'output_0': TensorSpec(shape=(None, 1), dtype=tf.float32, name='output_0')}


Menjalankan inferensi dengan memberikan input serialized examples ke serving signature:


In [5]:
input_tensor = tf.constant(serialized_examples)
predictions = serving_fn(examples=input_tensor)

output_key = list(predictions.keys())[0]
raw_probs = predictions[output_key].numpy().flatten()

print("Hasil Inferensi Model Serving:")
for i, (prob, sample) in enumerate(zip(raw_probs, samples)):
    pred_class = 1 if prob >= 0.5 else 0
    actual_class = int(sample['target'])
    status = "Risiko Penyakit Jantung (Positif)" if pred_class == 1 else "Normal / Sehat (Negatif)"
    
    print(f"\nPasien {i+1}:")
    print(f"  - Nilai Probabilitas : {prob:.4f}")
    print(f"  - Prediksi Kelas     : {pred_class} ({status})")
    print(f"  - Target Aktual      : {actual_class}")
    print(f"  - Status Validasi    : {'BENAR (Cocok)' if pred_class == actual_class else 'SALAH'}")


Hasil Inferensi Model Serving:

Pasien 1:
  - Nilai Probabilitas : 0.8728
  - Prediksi Kelas     : 1 (Risiko Penyakit Jantung (Positif))
  - Target Aktual      : 1
  - Status Validasi    : BENAR (Cocok)

Pasien 2:
  - Nilai Probabilitas : 0.0054
  - Prediksi Kelas     : 0 (Normal / Sehat (Negatif))
  - Target Aktual      : 0
  - Status Validasi    : BENAR (Cocok)


## 5. Pengujian 2: Format Prediction Request REST API (TensorFlow Serving)
Pada lingkungan produksi yang menjalankan TensorFlow Serving via Docker/REST API, klien mengirimkan permintaan HTTP POST dengan format JSON yang memuat serialized byte encoded dalam Base64.
Berikut adalah konstruksi payload request dan simulasi pemanggilannya.


In [6]:
# Membangun format REST API JSON Payload standar TensorFlow Serving
instances_payload = []
for ser in serialized_examples:
    instances_payload.append({
        "b64": base64.b64encode(ser).decode("utf-8")
    })

request_body = {
    "signature_name": "serving_default",
    "instances": instances_payload
}

print("Contoh Struktur Payload Request JSON (TensorFlow Serving REST API):")
print(json.dumps({
    "signature_name": request_body["signature_name"],
    "instances": [{"b64": instances_payload[0]["b64"][:50] + "... (truncated)"}]
}, indent=2))


Contoh Struktur Payload Request JSON (TensorFlow Serving REST API):
{
  "signature_name": "serving_default",
  "instances": [
    {
      "b64": "CtEBCgwKA2ZicxIFGgMKAQEKCwoCY3ASBRoDCgEDCg4KBXNsb3... (truncated)"
    }
  ]
}


Menguji endpoint TensorFlow Serving jika server aktif, atau menampilkan simulasi respons jika server offline:


In [7]:
TF_SERVING_URL = "http://localhost:8501/v1/models/heart-disease-model:predict"

try:
    headers = {"content-type": "application/json"}
    response = requests.post(TF_SERVING_URL, data=json.dumps(request_body), headers=headers, timeout=2)
    if response.status_code == 200:
        result = response.json()
        print("Respon Berhasil dari TensorFlow Serving REST API:")
        print(json.dumps(result, indent=2))
    else:
        print(f"Response status {response.status_code}: {response.text}")
except Exception as e:
    print(f"Catatan: TF Serving endpoint ({TF_SERVING_URL}) belum aktif di background.")
    print("Menampilkan struktur respon JSON standar yang dihasilkan:")
    simulated_response = {
        "predictions": [[float(p)] for p in raw_probs]
    }
    print(json.dumps(simulated_response, indent=2))


Catatan: TF Serving endpoint (http://localhost:8501/v1/models/heart-disease-model:predict) belum aktif di background.
Menampilkan struktur respon JSON standar yang dihasilkan:
{
  "predictions": [
    [
      0.8727744817733765
    ],
    [
      0.005380172282457352
    ]
  ]
}


## 6. Tabel Ringkasan Hasil Pengujian Prediksi


In [8]:
summary_data = []
for i, (prob, sample) in enumerate(zip(raw_probs, samples)):
    pred = 1 if prob >= 0.5 else 0
    actual = int(sample['target'])
    summary_data.append({
        "Pasien": f"Pasien {i+1}",
        "Umur": sample["age"],
        "Sex": "Laki-laki" if sample["sex"] == 1 else "Perempuan",
        "Detak Jantung (thalach)": sample["thalach"],
        "Probabilitas Prediksi": f"{prob:.4f}",
        "Kelas Prediksi": pred,
        "Kelas Aktual": actual,
        "Akurasi": "Sesuai (True)" if pred == actual else "Berbeda"
    })

summary_df = pd.DataFrame(summary_data)
summary_df


,Pasien,Umur,Sex,Detak Jantung (thalach),Probabilitas Prediksi,Kelas Prediksi,Kelas Aktual,Akurasi
0,Pasien 1,63.0,Laki-laki,150.0,0.8728,1,1,Sesuai (True)
1,Pasien 2,67.0,Laki-laki,108.0,0.0054,0,0,Sesuai (True)


---
## Kesimpulan Pengujian
1. Model serving yang dihasilkan oleh pipeline TFX telah berhasil diuji menggunakan signature `serving_default`.
2. Model mampu menerima input data mentah (*raw serialized example*), menerapkan preprocessing transform TFT secara otomatis, dan mengeluarkan probabilitas klasifikasi yang akurat.
3. Struktur format prediction request REST API siap dikonsumsi oleh TensorFlow Serving di lingkungan produksi.
